# SpatialAnnotator + FoundationGeo (Colab quickstart)

Runs the VQASynth SpatialAnnotator agent with **FoundationGeo v1.1** as the metric-depth backend, in place of the default DepthPro (CPU) or VGGT (GPU).

- Paper: [Learning Spatial Pixel-Wise Fields for Monocular Metric Geometry](https://arxiv.org/abs/2607.11588) (ECCV 2026)
- Model: [`mxliu-hku/FoundationGeo-1.1`](https://huggingface.co/mxliu-hku/FoundationGeo-1.1) (~314M params, metric)
- Requires a GPU runtime (T4 works). Set **Runtime → Change runtime type → T4 GPU** before running.
- Gemini API key: add `GEMINI_API_KEY` to Colab **Secrets** (left sidebar → key icon).

## 1. Install dependencies

Pins NumPy < 2 (VQASynth pipeline currently expects the 1.x API). **You must restart the runtime** after this cell before continuing — do it once when prompted, then start again from cell 2.

In [ ]:
!pip install -q 'numpy<2.0'
!pip install -q einops timm
!pip install -q 'git+https://github.com/EasternJournalist/utils3d.git'
!pip install -q 'git+https://github.com/mx-liu6/FoundationGeo.git'
!pip install -q 'git+https://github.com/remyxai/VQASynth.git@foundationgeo-learning-spatial-pixel-wise-fields-for-monocul'
!pip install -q 'nooa @ git+https://github.com/NVIDIA-NeMo/labs-OO-Agents.git@main' 'depth_pro @ git+https://github.com/apple/ml-depth-pro.git'

## 2. Setup — auth, imports, GPU check

In [ ]:
import os
from google.colab import userdata

# LiteLLM's Gemini adapter reads GEMINI_API_KEY for the `gemini/` prefix.
# Older versions checked GOOGLE_API_KEY — set both to cover.
_key = userdata.get('GEMINI_API_KEY').strip()
os.environ['GEMINI_API_KEY'] = _key
os.environ['GOOGLE_API_KEY'] = _key

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3. Build the agent with FoundationGeoEstimator

`SpatialAnnotator` picks a depth backend by tier (CPU → DepthPro, GPU → VGGT). To use FoundationGeo instead, instantiate the underlying agent class directly with a custom `depth` argument.

`FoundationGeoEstimator` accepts a `fov_x_deg` kwarg — if you know the source camera's horizontal FOV (e.g. from EXIF), pass it here. `None` lets FoundationGeo estimate it.

In [ ]:
from nooa.unifiedllm import get_llm_client
from experiments.nooa_agent.spatial_annotator import _make_agent_class
from experiments.nooa_agent.tools.florence import FlorenceDetector, FlorenceSegmenter
from experiments.nooa_agent.tools.depth import FoundationGeoEstimator

llm = get_llm_client('gemini/gemini-2.5-pro')

# fp16 on both — florence.py casts pixel_values to model.dtype before generate()
# so mixed-precision inference works cleanly.
detector = FlorenceDetector(device='cuda', dtype='fp16')
segmenter = FlorenceSegmenter(detector=detector)
depth = FoundationGeoEstimator(device='cuda', dtype='fp16')  # or fov_x_deg=<known FOV>

AgentCls = _make_agent_class('gpu', llm=llm)
agent = AgentCls(detector=detector, segmenter=segmenter, depth=depth)
print('agent ready; depth backend =', depth.__class__.__name__)

## 4. Load an image

Upload your own image, or use one from the FoundationGeo demo assets.

In [ ]:
import urllib.request
from PIL import Image

url = 'https://raw.githubusercontent.com/mx-liu6/FoundationGeo/main/demo/robotic.jpg'
urllib.request.urlretrieve(url, 'demo_scene.jpg')
image = Image.open('demo_scene.jpg').convert('RGB')
print(f'image size: {image.size}')
image

## 5. Ask a spatial question

The agent composes tool calls in code: detect objects → measure depth via FoundationGeo → compose geometry → answer. Try a distance/height/proximity question that requires real metric depth.

In [ ]:
question = 'What is the metric distance in meters between the two objects closest to the camera?'

result = await agent.annotate(image, question)

print('── Answer ──')
print(result.answer)
print(f'\nConfidence: {result.confidence}')
print(f'Tool calls used: {result.tool_calls_used}')
if result.supporting_evidence:
    print('\nSupporting evidence:')
    for e in result.supporting_evidence:
        print(f'  - {e}')

## 6. Inspect the depth backend directly

Sanity check — call `FoundationGeoEstimator.metric_depth` on the image and look at the depth statistics + returned intrinsics. FoundationGeo estimates the camera FOV internally if `fov_x_deg` wasn't passed; the intrinsics matrix reflects that estimate.

In [ ]:
import numpy as np

d = depth.metric_depth(image)
print(f'backend: {d.backend}')
print(f'depth shape: {d.depth_m.shape}, min/median/max = '
      f'{d.depth_m.min():.2f} / {np.median(d.depth_m):.2f} / {d.depth_m.max():.2f} m')
print(f'focal (px): {d.focal_px:.1f}')
print(f'intrinsics:\n{d.intrinsics_3x3}')
print(f'point cloud shape: {d.point_cloud_xyz.shape if d.point_cloud_xyz is not None else None}')

## 7. Serialize the trajectory (optional)

Every tool call + intermediate value is captured by NOOA. `qwen_vl_serialize` converts the trace into the Qwen2.5-VL / Qwen3-VL chat-template format for direct SFT.

In [ ]:
from experiments.nooa_agent.trace import capture_trace, qwen_vl_serialize

trace = capture_trace(agent)
messages = qwen_vl_serialize(trace, image=image, question=question, answer=result.answer)
for m in messages[:4]:
    role = m.get('role', '?')
    content = str(m.get('content', ''))[:200]
    print(f'[{role}] {content}...')

## Notes

- **FOV matters.** FoundationGeo's paper contribution is per-pixel corrections for focal-length OOD. If you know the source camera's FOV, pass `fov_x_deg=<value>` when constructing `FoundationGeoEstimator`. Otherwise the model estimates it, which is where zero-shot metric error accumulates.
- **Comparing backends.** Swap `FoundationGeoEstimator` for `VggtEstimator` or `DepthProEstimator` (all three implement `metric_depth(image) -> DepthResult`) and re-run cell 5 or 6 on the same image. Useful for triangulating scale on a known-dimension reference.
- **Resolution level.** `FoundationGeoEstimator(resolution_level=9)` is the paper default. Lower values (e.g. 5-7) speed up inference at some detail loss. Compare on the same query.
- **Composite license.** FoundationGeo's LICENSE is MIT (Microsoft upstream code) + Apache-2.0 (their additions). Compatible with VQASynth's Apache-2.0.